<a href="https://colab.research.google.com/github/vanderbilt-data-science/MNPSCollaborative/blob/New-Baseline-v2/mnps_new_baseline%20v7.7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **MNPS Job Equity New Baseline 7.7**
> A notebook to help you get started  
> DSI DSSG + MNPS   

> # **Version 7.7 Change**
> - **GPT-5-pro Compatible**: Proper API compatibility with capability probing and fallback
> - **Structured Outputs**: Uses JSON Schema for reliable GPT-5-pro responses
> - **Unified API Shim**: Single call_llm_json function that works for both GPT-4o and GPT-5-pro
> - **v7.5 Corrections**: Applies attribute-only corrections at the end
> - **Complete Pipeline**: All features from v7.1 with robust API handling


In [1]:
# ==== 1) Model Configuration for GPT-5-pro ====
import os
from google.colab import userdata

# 1) API key from Colab's 🔑 panel
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# 2) Read the model selector from Colab's 🔑 panel (can be alias or snapshot)
RAW_MODEL = userdata.get("OPENAI_MODEL")  # e.g., gpt-5-pro, gpt-4o, gpt-4o-2024-11-20, gpt4.1, o3 mini

def normalize_model_id(s: str | None) -> str | None:
    if not s:
        return None
    s = s.strip().lower().replace("_", "-").replace(" ", "-")
    return s

alias_or_snapshot = normalize_model_id(RAW_MODEL)

# 3) Map aliases → pinned snapshots you prefer (edit to taste)
SNAPSHOTS = {
    # GPT-5-pro (latest and most capable) - Note: Uses different API endpoint
    "gpt-5-pro": "gpt-5-pro",
    # GPT-4o snapshots (stable; good for Structured Outputs)
    "gpt-4o":  "gpt-4o-2024-11-20",
    # GPT-4.1 family snapshot (long context)
    "gpt-4.1": "gpt-4.1-2025-04-14",
    # Keep o3-mini as an alias (no public dated snapshot ID); good for reasoning
    "o3-mini": "o3-mini",
}

# 4) Final MODEL_ID selection rule:
#    - If user entered an alias, pin it via SNAPSHOTS
#    - If user entered a snapshot, pass it through
#    - Else fallback to a safe default snapshot
MODEL_ID = SNAPSHOTS.get(alias_or_snapshot or "", None) or (alias_or_snapshot) or "gpt-5-pro"

print("🔧 OPENAI_MODEL (raw):", RAW_MODEL)
print("✅ Using MODEL_ID:", MODEL_ID)


🔧 OPENAI_MODEL (raw): gpt-5-pro
✅ Using MODEL_ID: gpt-5-pro


## **2** | Environment Setup

### **2a** | API Key Setup
#### **2a.1** | Access
1. Click the 🔑 icon in the left sidebar
2. Add your OpenAI API key
3. Set `OPENAI_MODEL` to `gpt-5-pro` (or leave blank for default)

#### **2a.2** | API Keys in Google Colab
The notebook will automatically read your API key from the 🔑 panel.


In [25]:
# === Cell 2.0 — Model selection + GPT-5 safe call_llm_json (fixed caps object) ===
from types import SimpleNamespace
from dataclasses import dataclass
import json, re, os

# ---- Choose model here ----
MODEL_ID = os.environ.get("MODEL_ID", "gpt-5-pro")  # or "gpt-4o-2024-11-20"

# ---- OpenAI client (v1/v2 both OK) ----
try:
    import openai
    from openai import OpenAI
    client = OpenAI()
except Exception:
    # Older SDK fallback
    import openai
    client = openai

# ---- Capability probe returns *attributes*, not dict ----
def get_model_caps(model_id: str) -> SimpleNamespace:
    is_gpt5 = model_id.lower().startswith("gpt-5")
    is_responses = True  # we route through Responses API
    return SimpleNamespace(
        # used by downstream cells
        supports_parse_schema=is_gpt5,     # server-side structured output
        supports_create_schema=is_gpt5,    # ditto
        supports_json_schema=is_gpt5,      # your code checks this one specifically
        accepts_temperature=not is_gpt5,   # GPT-5 rejects temperature
        accepts_reasoning=is_gpt5,         # GPT-5 supports reasoning controls
        accepts_verbosity=is_gpt5,         # GPT-5 supports text.verbosity
        is_responses=is_responses
    )

caps = get_model_caps(MODEL_ID)

# ---- Safe Responses API wrapper (handles GPT-5 vs GPT-4o knobs) ----
def responses_create_safe(*, model: str, input=None, messages=None,
                          json_schema=None, max_output_tokens=1500):
    """
    - For GPT-5: no temperature; uses reasoning/text.verbosity/max_output_tokens
    - For GPT-4o (or others): passes temperature if provided via kwargs
    - If json_schema is provided and model supports it, asks for structured output
    """
    # Build content
    if input is None and messages is not None:
        # Convert ChatML-style messages to a single input array
        input = [{"role": m.get("role","user"), "content": m.get("content","")} for m in messages]
    if input is None:
        input = []

    # Base request
    req = {
        "model": model,
        "input": input,
    }

    # Structured output (only if model supports it and schema provided)
    if json_schema and getattr(caps, "supports_json_schema", False):
        req["response_format"] = {
            "type": "json_schema",
            "json_schema": {
                "name": "structured_output",
                "schema": json_schema,
                "strict": True
            }
        }
    else:
        # Ask for plain JSON text if no schema support
        req.setdefault("text_format", {"type": "plain"})

    # Controls
    if getattr(caps, "accepts_reasoning", False):
        req["reasoning"] = {"effort": "high"}
    if getattr(caps, "accepts_verbosity", False):
        req["text"] = {"verbosity": "high"}
    if max_output_tokens:
        req["max_output_tokens"] = int(max_output_tokens)

    # For GPT-4o or older, allow temperature if caller passes it
    # (we DO NOT include temperature by default to avoid breaking GPT-5)
    # Example: responses_create_safe(..., temperature=0.2)
    # will only be attached for non-GPT-5 models.
    def _attach_legacy_controls(req, **kwargs):
        if not getattr(caps, "accepts_temperature", False):
            kwargs.pop("temperature", None)
        req.update(kwargs)
        return req

    # If caller passed extra kwargs (like temperature), attach them carefully
    # You can extend this pattern as needed.
    req = _attach_legacy_controls(req)

    # Call API
    try:
        resp = client.responses.create(**req)
    except TypeError:
        # older SDK signature: use messages instead of input
        msgs = [{"role": x.get("role","user"), "content": x.get("content","")} for x in input]
        req2 = {k:v for k,v in req.items() if k != "input"}
        req2["messages"] = msgs
        resp = client.chat.completions.create(**req2)  # last-ditch fallback

    return resp

# ---- JSON convenience: returns parsed JSON when possible ----
def call_llm_json(prompt: str, *, model: str = None, json_schema: dict = None, max_output_tokens: int = 1500):
    model = model or MODEL_ID

    # Prefer structured if supported & schema provided
    if json_schema and getattr(caps, "supports_json_schema", False):
        resp = responses_create_safe(model=model, input=[{"role":"user","content": prompt}],
                                     json_schema=json_schema,
                                     max_output_tokens=max_output_tokens)
        # Most SDKs expose text; fall back to first output choice if needed
        try:
            txt = resp.output_text
        except Exception:
            txt = json.dumps(resp.dict() if hasattr(resp, "dict") else resp)
        # If the server enforced JSON schema, txt should already be valid JSON
        try:
            return json.loads(txt)
        except Exception:
            # Some SDKs return structured content differently; try to locate JSON
            m = re.search(r"\{.*\}\s*$", txt, flags=re.S)
            return json.loads(m.group(0)) if m else json.loads("{" + "}")

    # Otherwise plain text JSON (we’ll parse client-side)
    resp = responses_create_safe(model=model, input=[{"role":"user","content": prompt}],
                                 max_output_tokens=max_output_tokens)
    try:
        txt = resp.output_text
    except Exception:
        txt = str(resp)
    m = re.search(r"\{.*\}\s*$", txt, flags=re.S)
    if not m:
        # return raw text as last resort
        return {"_raw": txt}
    return json.loads(m.group(0))


In [26]:
# ===== Cell 3.3 — call_llm_json shim (GPT-5-Pro compatible, 4o fallback) =====
import json, re
from typing import Optional, Dict, Any

def call_llm_json(
    prompt: str,
    *,
    model: Optional[str] = None,
    schema: Optional[Dict[str, Any]] = None,   # JSON Schema (Responses Structured Outputs)
    temperature: float = 0.2,
    max_output_tokens: Optional[int] = None,
) -> str:
    """
    Returns a JSON string. If the model supports Structured Outputs, we enforce the schema.
    Otherwise, we coerce the model to emit JSON and extract it.
    """
    mid = model or globals().get("MODEL_ID", "gpt-4o-2024-11-20")
    caps = globals().get("CAPS", None)

    # If schema provided and model supports it → use Structured Outputs
    if schema and caps and caps.supports_json_schema:
        try:
            resp = client.responses.create(
                model=mid,
                input=[{"role":"user","content": prompt}],
                response_format={"type":"json_schema", "json_schema": {"name":"schema", "schema": schema}},
                temperature=temperature,
                max_output_tokens=max_output_tokens,
            )
            # Most SDKs expose a convenience accessor; fall back to raw text scrape if needed.
            try:
                return resp.output_text  # already JSON if structured outputs enforced
            except Exception:
                # scrape first {...} block from the response text
                txt = str(resp)
                m = re.search(r"\{.*\}", txt, flags=re.S)
                if not m:
                    raise ValueError("No JSON object found in structured response.")
                return m.group(0)
        except Exception as e:
            # fall through to plain mode with a warning
            print(f"[call_llm_json] Structured outputs unavailable or failed on '{mid}': {e}\nFalling back to plain JSON text mode.")

    # Plain JSON text mode (works on 4o and 5-Pro alike)
    sys_guard = (
        "Return ONLY valid JSON. No markdown. No commentary. "
        "If you cannot comply, return an empty JSON object {}."
    )
    txt_blocks = [
        {"role":"system","content": sys_guard},
        {"role":"user","content": prompt},
    ]
    resp = client.responses.create(
        model=mid,
        input=txt_blocks,
        temperature=temperature,
        max_output_tokens=max_output_tokens,
    )
    # Extract first JSON object from text
    try:
        out = resp.output_text
    except Exception:
        out = str(resp)
    m = re.search(r"\{.*\}", out, flags=re.S)
    if not m:
        # last resort: return an empty object so downstream never crashes
        return "{}"
    return m.group(0)

print("✅ call_llm_json shim defined")


✅ call_llm_json shim defined


In [27]:
# ===== Cell 3.4 — JSON Schemas for classification (GPT-5-Pro structured outputs) =====

# If VALID_ROLES is already built (e.g., by your 14.x roles loader), we'll
# enforce it as a closed set. Otherwise we fall back to a loose string.
_ROLE_ENUM = list(globals().get("VALID_ROLES", []))
_HAS_ENUM  = bool(_ROLE_ENUM)

SCHEMA_CLASSIFICATION_ROW = {
    "name": "mnps_classification_row",
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "required": [
            "new_job_title",
            "major_role_group",
            "minor_sub_group",
            "grouping_justification"
        ],
        "properties": {
            "new_job_title": {
                "type": "string",
                "minLength": 3,
                "maxLength": 200,
                "description": "Follow '[Function] [Role] [Level]' convention; do NOT echo the original title."
            },
            "major_role_group": (
                {"type": "string", "enum": _ROLE_ENUM}
                if _HAS_ENUM else
                {"type": "string", "minLength": 2}
            ),
            "minor_sub_group": {
                "type": "string",
                "enum": ["Lead", "I", "II", "III"],
                "description": "Approved levels only; never IV (map IV -> Lead if KSACs justify)."
            },
            "grouping_justification": {
                "type": "string",
                "minLength": 20,
                "maxLength": 2000,
                "description": "Cite KSACs/functions/education/experience/licensure ONLY. Never reference the job title."
            }
        }
    }
}

# For the batch Responses API path where you want a table plus an overall narrative:
SCHEMA_CLASSIFICATION_TABLE = {
    "name": "mnps_classification_table",
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "required": ["job_classification_table", "narrative_rationale"],
        "properties": {
            "job_classification_table": {
                "type": "array",
                "minItems": 1,
                "items": SCHEMA_CLASSIFICATION_ROW["schema"]
            },
            "narrative_rationale": {
                "type": "string",
                "minLength": 50,
                "maxLength": 4000,
                "description": "High-level reasoning and patterns; reference KSAC sources, NEVER titles."
            }
        }
    }
}

print("✅ JSON schemas defined for structured outputs")


✅ JSON schemas defined for structured outputs


In [28]:
# ===== Cell 3.5 responses_create_safe (use instead of raw client.responses.create) =====
def responses_create_safe(*, model=None, input=None, response_format=None,
                          temperature=None, max_output_tokens=None, **extra):
    mid = model or MODEL_ID
    is_g5 = str(mid).lower().startswith("gpt-5")
    kwargs = dict(model=mid, input=input, response_format=response_format)

    if is_g5:
        kwargs["reasoning"] = {"effort": REASONING_EFFORT}
        kwargs["text"] = {"verbosity": TEXT_VERBOSITY}
        kwargs["max_output_tokens"] = max_output_tokens or MAX_OUTPUT_TOKENS
        # DO NOT set temperature
    else:
        if temperature is not None:
            kwargs["temperature"] = float(temperature)
        if max_output_tokens is not None:
            kwargs["max_output_tokens"] = int(max_output_tokens)

    kwargs.update(extra)
    return client.responses.create(**kwargs)


## **3** | The Data

This notebook will use the same data structure as v7.1 but with GPT-5-pro inference.


In [29]:
# ==== Cell 4 — Unique run folder + get inputs (3 files) + robust CSV read + upload to OpenAI =====
import os, json, shutil, datetime as dt, zipfile
from pathlib import Path
import pandas as pd
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Create unique run folder
timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
run_folder = f"RUN_{timestamp}"
base_path = Path("/content/drive/My Drive/Colab Notebooks/Run Results")
run_path = base_path / run_folder
run_path.mkdir(parents=True, exist_ok=True)

# Create outputs subfolder
OUTPUTS_DIR = run_path / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Run folder: {run_path}")
print(f"📁 Outputs dir: {OUTPUTS_DIR}")

# Load all required files - Updated for Colab root path
RUN_ROOT = Path('/content')

# Unzip MNPS Prompt Resources if needed
ZIP_FILE = RUN_ROOT / "MNPS Prompt Resources.zip"
if ZIP_FILE.exists():
    print(f"📦 Found {ZIP_FILE}, extracting...")
    with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
        zip_ref.extractall(RUN_ROOT)
    print("✅ Extracted MNPS Prompt Resources")
else:
    print("⚠️  MNPS Prompt Resources.zip not found - make sure to upload it")

# Core data files
BATCH_INPUT_CSV = RUN_ROOT / "Sample JDs.csv"
GT_MASTERFILE_CSV = RUN_ROOT / "Ground Truth Masterfile.csv"

# MNPS Prompt Resources (from extracted zip)
MNPS_ROLES_CSV = RUN_ROOT / "MNPS Roles.csv"
MNPS_KSACS_CSV = RUN_ROOT / "MNPS KSACs.csv"
COMPETENCY_EXTENDED_CSV = RUN_ROOT / "Competency Extended Descriptions.csv"
KORN_FERRY_CSV = RUN_ROOT / "Korn_Ferry Lominger 38 Competencies.csv"

print(f"📄 Batch input: {BATCH_INPUT_CSV}")
print(f"📄 Ground truth: {GT_MASTERFILE_CSV}")
print(f"📄 MNPS roles: {MNPS_ROLES_CSV}")
print(f"📄 MNPS KSACs: {MNPS_KSACS_CSV}")
print(f"📄 Competency Extended: {COMPETENCY_EXTENDED_CSV}")
print(f"📄 Korn Ferry: {KORN_FERRY_CSV}")

# Load data
df = pd.read_csv(BATCH_INPUT_CSV, encoding='latin1')
gt_df = pd.read_csv(GT_MASTERFILE_CSV, encoding='latin1')
roles_df = pd.read_csv(MNPS_ROLES_CSV, encoding='latin1')
ksacs_df = pd.read_csv(MNPS_KSACS_CSV, encoding='latin1')
competency_df = pd.read_csv(COMPETENCY_EXTENDED_CSV, encoding='latin1')
korn_ferry_df = pd.read_csv(KORN_FERRY_CSV, encoding='latin1')


print(f"✅ Loaded {len(df)} job descriptions")
print(f"✅ Loaded {len(gt_df)} ground truth records")
print(f"✅ Loaded {len(roles_df)} MNPS roles")
print(f"✅ Loaded {len(ksacs_df)} MNPS KSACs")
print(f"✅ Loaded {len(competency_df)} competency descriptions")
print(f"✅ Loaded {len(korn_ferry_df)} Korn Ferry competencies")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📁 Run folder: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251014_183651
📁 Outputs dir: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251014_183651/outputs
📦 Found /content/MNPS Prompt Resources.zip, extracting...
✅ Extracted MNPS Prompt Resources
📄 Batch input: /content/Sample JDs.csv
📄 Ground truth: /content/Ground Truth Masterfile.csv
📄 MNPS roles: /content/MNPS Roles.csv
📄 MNPS KSACs: /content/MNPS KSACs.csv
📄 Competency Extended: /content/Competency Extended Descriptions.csv
📄 Korn Ferry: /content/Korn_Ferry Lominger 38 Competencies.csv
✅ Loaded 43 job descriptions
✅ Loaded 176 ground truth records
✅ Loaded 60 MNPS roles
✅ Loaded 300 MNPS KSACs
✅ Loaded 38 competency descriptions
✅ Loaded 38 Korn Ferry competencies


In [30]:
# ==== Cell 12 — Zero Shot Prompt (from v7.1) =====
zero_shot_prompt = \
""" Objective: Evaluate and group jobs from the "New Sample_08.07.2025.csv" file based on similarities in job functions, not job titles.

Process:

- Compare all jobs against each other using the attributes listed in the file: Education, Work Experience, Licenses/Certifications, Essential Functions, Knowledge, Skills, Abilities, and Position Summary.

- Group jobs that have similar functions, responsibilities, and requirements, regardless of their job titles.

- Use the attached reference sources (Ground Truth Masterfile, MNPS Roles, MNPS KSACs) to ensure alignment with MNPS standards and classifications.

- Focus on the actual work being performed, not the job title, to create meaningful and accurate groupings.

- Ensure that each grouping reflects the true nature of the work and aligns with MNPS role classifications and competency frameworks.

- Provide clear justification for each grouping decision based on the job attributes and MNPS standards.

- Never justify classifications based on job titles - only use job attributes and MNPS standards.

Output Requirements:
- Major Role Group: Choose from approved MNPS major role groupings
- Minor Sub Group: Use I, II, III, or Lead based on complexity and responsibility level
- Provide detailed justification based on job attributes and MNPS KSACs alignment"""

print("✅ Zero shot prompt defined")


✅ Zero shot prompt defined


In [31]:
# ==== Cell 15.0 — Role Confidence Output Schema (v6.0) =====
from pydantic import BaseModel, Field
from typing import List, Optional

class RoleConfidenceTable(BaseModel):
    """Schema for role confidence evaluation output."""
    role: str = Field(description="The MNPS role name")
    confidence: float = Field(description="Confidence score 0.0-1.0")
    reasoning: str = Field(description="Brief explanation of the confidence score")

class RoleConfidenceResponse(BaseModel):
    """Response containing role confidence evaluations."""
    evaluations: List[RoleConfidenceTable] = Field(description="List of role confidence evaluations")

print("✅ Role confidence schema defined")


✅ Role confidence schema defined


In [32]:
# ==== Cell 15.1 — Role Confidence Prompt Builder (v6.0) =====
def build_role_confidence_prompt(job_description: str, mnps_roles: List[str], ksacs: str) -> str:
    """Build prompt for role confidence evaluation."""
    roles_list = "\n".join([f"- {role}" for role in mnps_roles])

    prompt = f"""You are an expert job classification system for Metro Nashville Public Schools (MNPS).

Your task is to evaluate how well a job description matches each MNPS role based on the job attributes (Position Summary, Essential Functions, Work Experience, Education, Licenses and Certifications, Knowledge, Skills and Abilities).

**IMPORTANT**: Ignore the job title completely. Base your evaluation solely on the job attributes.

Available MNPS Roles:
{roles_list}

MNPS Knowledge, Skills, Abilities, and Competencies (KSACs):
{ksacs}

Job Description:
{job_description}

For each MNPS role, provide:
1. A confidence score (0.0-1.0) indicating how well the job description matches the role
2. Brief reasoning for your confidence score

Return your response as a JSON object with the following structure:
{{
  "evaluations": [
    {{
      "role": "Role Name",
      "confidence": 0.85,
      "reasoning": "Brief explanation"
    }}
  ]
}}

Evaluate ALL roles listed above."""

    return prompt

print("✅ Role confidence prompt builder defined")


✅ Role confidence prompt builder defined


In [33]:
# ==== Cell 15.2 — Role Confidence Shortlist (robust build + canonicalize + write; empty-safe) =====

# Get MNPS roles and KSACs
VALID_ROLES = roles_df['Roles'].tolist() # Corrected column name

# Build comprehensive KSACs text from all resources
def build_ksacs_text():
    """Build comprehensive KSACs text from all MNPS resources."""
    ksacs_text = "MNPS Knowledge, Skills, Abilities, and Competencies (KSACs):\n\n"

    # Clean up column names to handle potential whitespace or case issues
    ksacs_df.columns = ksacs_df.columns.str.strip()
    competency_df.columns = competency_df.columns.str.strip()
    korn_ferry_df.columns = korn_ferry_df.columns.str.strip()


    # Add role-specific KSACs
    # Find columns that contain 'Role' and 'KSACs' (case-insensitive and partial match)
    role_col_ksacs = next((col for col in ksacs_df.columns if 'role' in col.lower()), None)
    ksacs_col_ksacs = next((col for col in ksacs_df.columns if 'ksacs' in col.lower()), None)

    if role_col_ksacs and ksacs_col_ksacs:
        for _, row in ksacs_df.iterrows():
            role = row.get(role_col_ksacs, '')
            ksacs = row.get(ksacs_col_ksacs, '')
            if role and ksacs:
                ksacs_text += f"**{role}**:\n{ksacs}\n\n"
    else:
        print("Warning: Could not find 'Role' or 'KSACs' columns in ksacs_df.")


    # Add competency extended descriptions
    # Find columns that contain 'Competency' and 'Description' (case-insensitive and partial match)
    comp_col_comp = next((col for col in competency_df.columns if 'competency' in col.lower()), None)
    desc_col_comp = next((col for col in competency_df.columns if 'description' in col.lower()), None)

    if comp_col_comp and desc_col_comp:
        ksacs_text += "\n**Competency Extended Descriptions**:\n"
        for _, row in competency_df.iterrows():
            competency = row.get(comp_col_comp, '')
            description = row.get(desc_col_comp, '')
            if competency and description:
                ksacs_text += f"- {competency}: {description}\n"
    else:
         print("Warning: Could not find 'Competency' or 'Description' columns in competency_df.")

    # Add Korn Ferry competencies
    # Find columns that contain 'Competency' and 'Definition' (case-insensitive and partial match)
    comp_col_kf = next((col for col in korn_ferry_df.columns if 'competency' in col.lower()), None)
    def_col_kf = next((col for col in korn_ferry_df.columns if 'description' in col.lower() or 'definition' in col.lower()), None)


    if comp_col_kf and def_col_kf:
        ksacs_text += "\n**Korn Ferry Lominger 38 Competencies**:\n"
        for _, row in korn_ferry_df.iterrows():
            competency = row.get(comp_col_kf, '')
            definition = row.get(def_col_kf, '')
            if competency and definition:
                ksacs_text += f"- {competency}: {definition}\n"
    else:
        print("Warning: Could not find 'Competency' or 'Description'/'Definition' columns in korn_ferry_df.")


    return ksacs_text

KSACS_TEXT = build_ksacs_text()

print(f"✅ Found {len(VALID_ROLES)} MNPS roles")
print(f"✅ Built comprehensive KSACs text ({len(KSACS_TEXT)} characters)")
print(f"✅ Using model: {MODEL_ID}")

# Update the schema with the loaded roles
_ROLE_ENUM = list(VALID_ROLES)
_HAS_ENUM = bool(_ROLE_ENUM)

# Rebuild the schema with the actual roles
SCHEMA_CLASSIFICATION_ROW = {
    "name": "mnps_classification_row",
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "required": [
            "new_job_title",
            "major_role_group",
            "minor_sub_group",
            "grouping_justification"
        ],
        "properties": {
            "new_job_title": {
                "type": "string",
                "minLength": 3,
                "maxLength": 200,
                "description": "Follow '[Function] [Role] [Level]' convention; do NOT echo the original title."
            },
            "major_role_group": (
                {"type": "string", "enum": _ROLE_ENUM}
                if _HAS_ENUM else
                {"type": "string", "minLength": 2}
            ),
            "minor_sub_group": {
                "type": "string",
                "enum": ["Lead", "I", "II", "III"],
                "description": "Approved levels only; never IV (map IV -> Lead if KSACs justify)."
            },
            "grouping_justification": {
                "type": "string",
                "minLength": 20,
                "maxLength": 2000,
                "description": "Cite KSACs/functions/education/experience/licensure ONLY. Never reference the job title."
            }
        }
    }
}

print("✅ Updated schema with loaded MNPS roles")

✅ Found 60 MNPS roles
✅ Built comprehensive KSACs text (37794 characters)
✅ Using model: gpt-5-pro
✅ Updated schema with loaded MNPS roles


In [34]:
# ==== Cell 16 — Batch Processing with GPT-5-pro (Structured Outputs) =====
import time
from tqdm import tqdm

def process_job_description(row_idx: int, row: pd.Series) -> dict:
    """Process a single job description using structured outputs."""
    # Build job description text (ignore job title)
    job_text = f"""Position Summary: {row.get('Position Summary', '')}
Essential Functions: {row.get('Essential Functions', '')}
Work Experience: {row.get('Work Experience', '')}
Education: {row.get('Education', '')}
Licenses and Certifications: {row.get('Licenses and Certifications', '')}
Knowledge, Skills and Abilities: {row.get('Knowledge, Skills and Abilities', '')}"""

    # Build comprehensive prompt
    prompt = f"""{zero_shot_prompt}

Available MNPS Roles: {', '.join(VALID_ROLES)}

MNPS Knowledge, Skills, Abilities, and Competencies (KSACs):
{KSACS_TEXT}

Job Description to Classify:
{job_text}

**IMPORTANT**:
- Ignore the job title completely
- Base classification solely on job attributes
- Use only approved MNPS roles and levels (I, II, III, Lead)
- Provide detailed justification based on KSACs alignment"""

    try:
        # Use structured outputs with the schema
        raw_response = call_llm_json(
            prompt,
            model=MODEL_ID,
            schema=SCHEMA_CLASSIFICATION_ROW["schema"],
            temperature=0.2
        )

        # Parse the JSON response
        response_data = json.loads(raw_response)

        return {
            'source_row_index': row_idx,
            'job_title_original': row.get('Job Title', ''),
            'new_job_title': response_data.get('new_job_title', 'Unknown'),
            'major_role_group': response_data.get('major_role_group', 'Other'),
            'minor_sub_group': response_data.get('minor_sub_group', 'I'),
            'grouping_justification': response_data.get('grouping_justification', 'No justification provided'),
            'model_used': MODEL_ID
        }
    except Exception as e:
        print(f"Error processing row {row_idx}: {e}")
        return {
            'source_row_index': row_idx,
            'job_title_original': row.get('Job Title', ''),
            'new_job_title': 'Error',
            'major_role_group': 'Other',
            'minor_sub_group': 'I',
            'grouping_justification': f'Error: {str(e)}',
            'model_used': MODEL_ID
        }

# Process all job descriptions
results = []
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing jobs"):
    result = process_job_description(idx, row)
    results.append(result)
    time.sleep(0.1)  # Rate limiting

# Save results
results_df = pd.DataFrame(results)
output_path = OUTPUTS_DIR / "Job_Classifications_Batch_gpt5pro_structured.csv"
results_df.to_csv(output_path, index=False)

print(f"✅ Processed {len(results)} job descriptions")
print(f"✅ Saved results to: {output_path}")


Processing jobs:   2%|▏         | 1/43 [00:00<00:04,  9.90it/s]

Error processing row 0: 'Caps' object has no attribute 'supports_json_schema'
Error processing row 1: 'Caps' object has no attribute 'supports_json_schema'


Processing jobs:   7%|▋         | 3/43 [00:00<00:04,  9.83it/s]

Error processing row 2: 'Caps' object has no attribute 'supports_json_schema'
Error processing row 3: 'Caps' object has no attribute 'supports_json_schema'


Processing jobs:  12%|█▏        | 5/43 [00:00<00:03,  9.82it/s]

...

Error processing row 38: 'Caps' object has no attribute 'supports_json_schema'
Error processing row 39: 'Caps' object has no attribute 'supports_json_schema'


Processing jobs:  95%|█████████▌| 41/43 [00:04<00:00,  9.81it/s]

Error processing row 40: 'Caps' object has no attribute 'supports_json_schema'
Error processing row 41: 'Caps' object has no attribute 'supports_json_schema'


Processing jobs: 100%|██████████| 43/43 [00:04<00:00,  9.81it/s]

Error processing row 42: 'Caps' object has no attribute 'supports_json_schema'
✅ Processed 43 job descriptions
✅ Saved results to: /content/drive/My Drive/Colab Notebooks/Run Results/RUN_20251014_183651/outputs/Job_Classifications_Batch_gpt5pro_structured.csv


## **4** | v7.5 Corrections Applied

Now apply the same corrections from v7.5 to the GPT-5-pro results:


In [ ]:
# ==== v7.5 Corrections for GPT-5-pro Results =====
import re
import numpy as np

# Load the GPT-5-pro results
preds = results_df.copy()
attrs = df.copy()

# Closed sets and normalization helpers
MAJOR_ALLOWED = [
    'Technician','Specialist','Analyst','Manager','Coordinator','Director','Other',
    'Teacher','Coach','Counselor','Clerical Support','Instructor','Driver'
]
MINOR_ALLOWED = ['I','II','III','Lead']

CANON_MINOR_MAP = {
    'i':'I','1':'I','one':'I','entry':'I',
    'ii':'II','2':'II','two':'II',
    'iii':'III','3':'III','three':'III',
    'lead':'Lead','iv':'III','4':'III'
}

SPECIALIST_FALLBACKS = [
    ('Teacher','classroom|lesson|instruction|teacher|students'),
    ('Coach','coach|instructional coach|plc|model lessons|co-teach'),
    ('Clerical Support','clerk|clerical|records|data entry|office support'),
    ('Counselor','counsel|social-emotional|guidance'),
    ('Manager','manage|supervise|budget|oversight|lead team|program manager'),
]

def normalize_minor(x: str) -> str:
    if pd.isna(x):
        return 'I'
    s = str(x).strip()
    if s in MINOR_ALLOWED:
        return s
    s_low = s.lower()
    return CANON_MINOR_MAP.get(s_low, 'I')

def discourage_specialist(text: str, proposed_major: str) -> str:
    if proposed_major != 'Specialist':
        return proposed_major
    t = (text or '').lower()
    for major, pattern in SPECIALIST_FALLBACKS:
        if re.search(pattern, t):
            return major
    return proposed_major

print("✅ v7.5 correction functions defined")


In [ ]:
# ==== Apply v7.5 Corrections =====

# Build attribute-only text
ATTR_COLS = [
    'Position Summary','Essential Functions','Work Experience','Education',
    'Licenses and Certifications','Knowledge, Skills and Abilities'
]

text = (
    attrs['Position Summary'].fillna('') + ' ' +
    attrs['Essential Functions'].fillna('') + ' ' +
    attrs['Work Experience'].fillna('') + ' ' +
    attrs['Education'].fillna('') + ' ' +
    attrs['Licenses and Certifications'].fillna('') + ' ' +
    attrs['Knowledge, Skills and Abilities'].fillna('')
)

# Apply corrections
maj0 = preds.get('major_role_group', pd.Series(['Other']*len(preds)))
min0 = preds.get('minor_sub_group', pd.Series(['I']*len(preds)))

ref_major = []
for i, m in enumerate(maj0):
    proposed = str(m) if pd.notna(m) else 'Other'
    proposed = proposed if proposed in MAJOR_ALLOWED else 'Other'
    proposed = discourage_specialist(text.iloc[i], proposed)
    ref_major.append(proposed)

ref_minor = [normalize_minor(x) for x in min0]

# Create corrected results
corrected = preds.copy()
corrected['major_role_group'] = ref_major
corrected['minor_sub_group'] = ref_minor

# Save corrected results
corrected_path = OUTPUTS_DIR / "Job_Classifications_Batch_gpt5pro_v75_corrected.csv"
corrected.to_csv(corrected_path, index=False)

print(f"✅ Applied v7.5 corrections")
print(f"✅ Saved corrected results to: {corrected_path}")


In [ ]:
# ==== Generate Summary Statistics =====

before_major = preds.get('major_role_group', pd.Series(['']*len(preds))).astype(str)
before_minor = preds.get('minor_sub_group', pd.Series(['']*len(preds))).astype(str)
after_major  = corrected['major_role_group'].astype(str)
after_minor  = corrected['minor_sub_group'].astype(str)

counts = pd.DataFrame({
    'key': ['rows','major_changed','minor_changed','specialist_after_count'],
    'value': [
        len(corrected),
        int((before_major!=after_major).sum()),
        int((before_minor!=after_minor).sum()),
        int((after_major=='Specialist').sum())
    ]
})

counts_path = OUTPUTS_DIR / "correction_counts_gpt5pro_structured.csv"
counts.to_csv(counts_path, index=False)

# Show examples of changes
ex_idx = ((before_major!=after_major) | (before_minor!=after_minor)).to_numpy().nonzero()[0][:6]
examples = pd.DataFrame({
    'row': ex_idx,
    'job_title_original': preds['job_title_original'].iloc[ex_idx],
    'major_before': before_major.iloc[ex_idx],
    'major_after': after_major.iloc[ex_idx],
    'minor_before': before_minor.iloc[ex_idx],
    'minor_after': after_minor.iloc[ex_idx],
})

examples_path = OUTPUTS_DIR / "examples_gpt5pro_structured.csv"
examples.to_csv(examples_path, index=False)

print("\n📊 Summary Statistics:")
print(counts.to_string(index=False))

print("\n📝 Example Corrections:")
print(examples.to_string(index=False))

print(f"\n✅ Saved counts to: {counts_path}")
print(f"✅ Saved examples to: {examples_path}")
